In [5]:
# Install the libraries used in this notebook.
!pip install pandas requests folium python-dotenv

In [23]:
# Import the libraries and load the TomTom API key.

import os
import requests
import pandas as pd
import folium

from dotenv import load_dotenv
load_dotenv(r"C:\Users\monis\Traffic API project\.env")
TOMTOM_API_KEY = os.getenv("TOMTOM_API_KEY")

# Define the NYC locations to analyze.

NYC_POINTS = [
    {"name": "Times Square", "lat": 40.7580, "lng": -73.9855},
    {"name": "Central Park South", "lat": 40.7651, "lng": -73.9769},
    {"name": "Grand Central", "lat": 40.7527, "lng": -73.9772},
    {"name": "Wall Street", "lat": 40.7066, "lng": -74.0090},
    {"name": "Brooklyn Bridge", "lat": 40.7061, "lng": -73.9969},
]

In [25]:
# Retrieve live traffic data from the TomTom Traffic API.

def get_flow(lat, lng):
    url = (
        "https://api.tomtom.com/traffic/services/4/"
        f"flowSegmentData/relative0/10/json?point={lat},{lng}&key={TOMTOM_API_KEY}"
    )

    data = requests.get(url).json()

    return data["flowSegmentData"]

In [27]:
# Collect traffic data for each NYC location.

rows = []

for p in NYC_POINTS:
    d = get_flow(p["lat"], p["lng"])

    if not d:
        print(f"No data for {p['name']}")
        continue

    rows.append({
        "name": p["name"],
        "lat": p["lat"],
        "lng": p["lng"],
        "currentSpeed": d.get("currentSpeed"),
        "freeFlowSpeed": d.get("freeFlowSpeed"),
        "currentTravelTime": d.get("currentTravelTime"),
        "freeFlowTravelTime": d.get("freeFlowTravelTime"),
        "confidence": d.get("confidence"),
    })

df = pd.DataFrame(rows)
df

,name,lat,lng,currentSpeed,freeFlowSpeed,currentTravelTime,freeFlowTravelTime,confidence
0,Times Square,40.7580,-73.9855,5,9,286,159,1.000000
1,Central Park South,40.7651,-73.9769,11,11,147,147,1.000000
2,Grand Central,40.7527,-73.9772,11,13,375,317,1.000000
3,Wall Street,40.7066,-74.0090,5,15,645,215,0.966496
4,Brooklyn Bridge,40.7061,-73.9969,15,43,567,198,1.000000


In [28]:
# Calculate the congestion ratio for each location.
df["congestion_ratio"] = df["currentSpeed"] / df["freeFlowSpeed"]
df


,name,lat,lng,currentSpeed,freeFlowSpeed,currentTravelTime,freeFlowTravelTime,confidence,congestion_ratio
0,Times Square,40.7580,-73.9855,5,9,286,159,1.000000,0.555556
1,Central Park South,40.7651,-73.9769,11,11,147,147,1.000000,1.000000
2,Grand Central,40.7527,-73.9772,11,13,375,317,1.000000,0.846154
3,Wall Street,40.7066,-74.0090,5,15,645,215,0.966496,0.333333
4,Brooklyn Bridge,40.7061,-73.9969,15,43,567,198,1.000000,0.348837


In [43]:
# Retrieve weather and air quality data from the Open-Meteo API.
import requests

lat, lon = 40.733, -73.995  # Manhattan center

# Weather forecast for next 6 hours
wx_url = "https://api.open-meteo.com/v1/forecast"
wx_params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "precipitation_probability,wind_gusts_10m,weather_code",
    "forecast_days": 1
}
wx = requests.get(wx_url, params=wx_params).json()

# Air quality forecast
air_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
air_params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "pm2_5,us_aqi",
    "forecast_days": 1,
    "timezone": "auto"
}
air = requests.get(air_url, params=air_params).json()


In [54]:
# Extract the weather values used to calculate travel risk.
rain = wx["hourly"]["precipitation_probability"][0]
wind = wx["hourly"]["wind_gusts_10m"][0]
weather_code = wx["hourly"]["weather_code"][0]
aqi = air["hourly"]["us_aqi"][0]

print("Rain chance:", rain, "%")
print("Wind gusts:", wind, "km/h")
print("Air Quality Index (AQI):", aqi)

Rain chance: 22 %
Wind gusts: 20.5 km/h
Air Quality Index (AQI): 53


In [56]:
# Convert the official Open-Meteo weather codes into easy-to-read weather icons.

if weather_code == 0:
    weather_icon = "☀️"
elif weather_code in [1, 2]:
    weather_icon = "🌤️"
elif weather_code == 3:
    weather_icon = "☁️"
elif weather_code in [45, 48]:
    weather_icon = "🌫️"
elif weather_code in [51, 53, 55, 61, 63, 65, 80, 81, 82]:
    weather_icon = "🌧️"
elif weather_code in [71, 73, 75, 77, 85, 86]:
    weather_icon = "❄️"
elif weather_code in [95, 96, 99]:
    weather_icon = "⛈️"
else:
    weather_icon = "🌡️"


Current weather: ☁️


In [72]:
# Convert the official Open-Meteo weather code into a weather description.

weather_descriptions = {
    0: "Clear Sky",
    1: "Mainly Clear",
    2: "Partly Cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Fog",
    51: "Light Drizzle",
    53: "Moderate Drizzle",
    55: "Heavy Drizzle",
    61: "Light Rain",
    63: "Moderate Rain",
    65: "Heavy Rain",
    71: "Light Snow",
    73: "Moderate Snow",
    75: "Heavy Snow",
    80: "Rain Showers",
    81: "Rain Showers",
    82: "Heavy Rain Showers",
    95: "Thunderstorm",
    96: "Thunderstorm with Hail",
    99: "Thunderstorm with Hail"
}

weather_description = weather_descriptions.get(weather_code, "Unknown")

In [74]:
# Display the current weather at each NYC location.

weather_map = folium.Map(location=[40.74, -73.99], zoom_start=12)

# Add a weather icon to each location.
for _, row in df.iterrows():

    # Create the popup shown when the weather icon is clicked.
    popup = folium.Popup(
        f"📍 {row['name']}<br><br>"
        f"{weather_icon} {weather_description}<br>"
        f"🌧️ Rain chance: {rain}%<br>"
        f"💨 Wind gusts: {wind} km/h<br>"
        f"🌫️ Air Quality Index: {aqi}",
        max_width=300
    )

    # Place the weather icon on the map.
    folium.Marker(
        location=[row["lat"], row["lng"]],
        popup=popup,
        icon=folium.DivIcon(
            html=f"<div style='font-size:30px;'>{weather_icon}</div>"
        )
    ).add_to(weather_map)

# Display the weather map.
weather_map

In [58]:
# Calculate the average traffic risk across all locations.

df["traffic_risk"] = (1 - df["congestion_ratio"]) * 100

city_risk = df["traffic_risk"].mean()

In [60]:
# Calculate the overall travel risk score.
weather_risk = (rain + (min(wind, 60) / 60) * 100) / 2
air_risk = min(aqi, 200) / 2

overall_risk = round(
    0.5 * city_risk +
    0.3 * weather_risk +
    0.2 * air_risk,
    1
)

print("Traffic risk:", round(city_risk, 1))
print("Weather risk:", round(weather_risk, 1))
print("Air quality risk:", round(air_risk, 1))
print("Overall travel risk:", overall_risk)

Traffic risk: 38.3
Weather risk: 28.1
Air quality risk: 26.5
Overall travel risk: 32.9
